# Drive → VideoEdit

在 Colab 的雲端 CPU runtime 執行，逐支讀 Italy 和 Rome 的原始 MP4。成功後會在既有私有 GitHub repo 裡發佈影片附件，並在 videoedit 建立索引。執行失敗可重跑，已核對的附件會跳過。

1. 如果 Drive 裡看不到 Italy 和 Rome，在 Google Drive 把兩個資料夾加到「我的雲端硬碟」的捷徑，或在第一格填實際位置。
2. 在 Colab 左側 Secrets 新增 VIDEOEDIT_TOKEN。使用只限 AriesHongHuanWu/VideoEdit 的 GitHub fine-grained token，Contents 設為 Read and write，並開啟這份 notebook 的存取。請勿把憑證貼到程式格或聊天。
3. 依序執行兩格。第一格會請你在 Google 官方視窗授權唯讀掛載。第二格會檢查 252 支檔案的大小後開始轉存。

In [ ]:
from google.colab import drive
drive.mount("/content/drive", readonly=True)

from pathlib import Path
paths = {
    "Italy": Path("/content/drive/MyDrive/Italy"),
    "Rome": Path("/content/drive/MyDrive/Rome"),
}
for label, path in paths.items():
    print(label, "OK" if path.is_dir() else "找不到，請更新這個路徑")

In [ ]:
import base64
import os
from pathlib import Path
import subprocess
import sys
import tempfile

from google.colab import userdata

assert all(path.is_dir() for path in paths.values()), "先確認 Italy 和 Rome 路徑"
token = userdata.get("VIDEOEDIT_TOKEN")
assert token, "在 Colab Secrets 加入 VIDEOEDIT_TOKEN"
env = dict(os.environ)
env["GH_TOKEN"] = token
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
basic = base64.b64encode(("x-access-token:" + token).encode()).decode()
env["GIT_CONFIG_VALUE_0"] = "AUTHORIZATION: basic " + basic
env["TARGET_BRANCH"] = "claude/relaxed-planck-dprxpx"
for group, path in paths.items():
    env[f"COLAB_DRIVE_{group.upper()}"] = str(path)

with tempfile.TemporaryDirectory(dir="/content", prefix="videoedit-transfer-") as cloud_path:
    checkout = Path(cloud_path) / "repo"
    subprocess.run(["git", "clone", "--single-branch", "--branch",
                    "codex/import-italy-rome-20261126",
                    "https://github.com/AriesHongHuanWu/VideoEdit.git", str(checkout)],
                   env=env, check=True)
    subprocess.run([sys.executable, ".github/drive-video-import/import.py"],
                   cwd=checkout, env=env, check=True)
